In [1]:
import pandas as pd
import matplotlib
import ast
#matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

#폰트 설정(없으면 영문으로 나옴)
try:
    font_path = [f for f in fm.findSystemFonts() if 'NanumGothic' in f or 'Malgun' in f or 'AppleGothic' in f]
    if font_path:
        plt.rcParams['font.family'] = fm.FontProperties(fname=font_path[0]).get_name()
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False




In [2]:
#--------------------
# 데이터 로드 함수
#--------------------   

def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

portfolio = load_data('../dataset/portfolio.csv')
profile = load_data('../dataset/profile.csv')
transcript = load_data('../dataset/transcript.csv')

portfolio.head()

,reward,channels,difficulty,duration,offer_type,id
0,10,"['email', 'mobile', 'social']",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"['web', 'email', 'mobile', 'social']",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"['web', 'email', 'mobile']",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"['web', 'email', 'mobile']",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"['web', 'email']",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [3]:
profile.head()


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


In [4]:
profile['became_member_on'].dtype

dtype('int64')

In [5]:
profile['became_member_on'].min(), profile['became_member_on'].max()

(np.int64(20130729), np.int64(20180726))

In [6]:
transcript.head()

,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [7]:
portfolio.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   reward      10 non-null     int64
 1   channels    10 non-null     str  
 2   difficulty  10 non-null     int64
 3   duration    10 non-null     int64
 4   offer_type  10 non-null     str  
 5   id          10 non-null     str  
dtypes: int64(3), str(3)
memory usage: 612.0 bytes


In [8]:
transcript.info()

<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   person  306534 non-null  str  
 1   event   306534 non-null  str  
 2   value   306534 non-null  str  
 3   time    306534 non-null  int64
dtypes: int64(1), str(3)
memory usage: 9.4 MB


In [9]:
profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               17000 non-null  int64  
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 664.2 KB


In [10]:
profile['income'].describe()

count     14825.000000
mean      65404.991568
std       21598.299410
min       30000.000000
25%       49000.000000
50%       64000.000000
75%       80000.000000
max      120000.000000
Name: income, dtype: float64

In [11]:
profile['income'].isnull().sum()

np.int64(2175)

In [12]:
# ----------------------
# 2. portfolio 전처리
# ----------------------

# channels 리스트 문자열 → 이진 4컬럼
portfolio['channels_list'] = portfolio['channels'].apply(ast.literal_eval)

for ch in ['web', 'email', 'mobile', 'social']:
    portfolio[f'ch_{ch}'] = portfolio['channels_list'].apply(
        lambda x: 1 if ch in x else 0
    )

# 채널 수 (몇 개 채널로 발송됐는지)
portfolio['channel_count'] = (
    portfolio['ch_web'] + portfolio['ch_email'] +
    portfolio['ch_mobile'] + portfolio['ch_social']
)

portfolio.drop(columns=['channels', 'channels_list'], inplace=True)

print("✅ portfolio 전처리 완료")
portfolio.head()

✅ portfolio 전처리 완료


,reward,difficulty,duration,offer_type,id,ch_web,ch_email,ch_mobile,ch_social,channel_count
0,10,10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,0,1,1,1,3
1,10,10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1,4
2,0,0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,1,0,3
3,5,5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,1,0,3
4,5,20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,1,0,0,2


In [13]:
#-----------------------
# 3. profile 전처리
#-----------------------

# age 컬럼: 100세 초과 → 결측치
profile['age'] = profile['age'].where(profile['age'] <= 100, other=None)

#became_member_on → datetime 변경
profile['became_member_on'] = pd.to_datetime(profile['became_member_on'].astype(str), format='%Y%m%d')

#멤버쉽 기간(일수): 데이터 기준일 = 가입일 + 1일 (가입일 포함하려고)
#min: 2013-07-29
#max: 2018-07-26
ref_date = profile['became_member_on'].max() + pd.Timedelta(days=1)
profile['member_days'] = (ref_date - profile['became_member_on']).dt.days

#연령대 파생변수로 
def age_group(age):
    if pd.isna(age):
        return '누락'
    elif age < 20:
        return '20대 미만'
    elif age < 30:
        return '20대'
    elif age < 40:
        return '30대'
    elif age < 50:
        return '40대'
    elif age < 60:
        return '50대'
    else:
        return '60대 이상'

profile['age_group'] = profile['age'].apply(age_group)

#소득구간 파생변수로
# income mean:65404.9
# income min: 30000.0
# income max: 120000.0
def income_group(income):
    if pd.isna(income):
        return '누락'
    elif income < 50000:
        return '5만 미만'
    elif income < 75000:
        return '5-7.5만'
    elif income < 100000:
        return '7.5-10만'
    else:
        return '10만 이상'
profile['income_group'] = profile['income'].apply(income_group)


#멤버십 구간 기간 파생변수로
def member_period(days):
    if pd.isna(days) or days < 0:
        return '미기재'
    elif days < 365:
        return '신규 (1년 미만)'
    elif days < 730:
        return '중간 (1~2년)'
    else:
        return '장기 (2년+)'

profile['member_days_group'] = profile['member_days'].apply(member_period)

In [14]:
profile.head()

,gender,age,id,became_member_on,income,member_days,age_group,income_group,member_days_group
0,NaN,NaN,68be06ca386d4c31939f3a4f0e3dd783,2017-02-12,NaN,530,누락,누락,중간 (1~2년)
1,F,55.0,0610b486422d4921ae7d2bf64640c50b,2017-07-15,112000.0,377,50대,10만 이상,중간 (1~2년)
2,NaN,NaN,38fe809add3b4fcf9315a9694bb96ff5,2018-07-12,NaN,15,누락,누락,신규 (1년 미만)
3,F,75.0,78afa995795e4d85b5d9ceeca43f5fef,2017-05-09,100000.0,444,60대 이상,10만 이상,중간 (1~2년)
4,NaN,NaN,a03223e636434f42ac4c3df47e8bac43,2017-08-04,NaN,357,누락,누락,신규 (1년 미만)


In [15]:
profile.isnull().sum()

gender               2175
age                  2180
id                      0
became_member_on        0
income               2175
member_days             0
age_group               0
income_group            0
member_days_group       0
dtype: int64

In [16]:
#-----------------------
# 4. transcript 전처리
#-----------------------
# 주의: received/viewed → 'offer id' (공백)
#       completed → 'offer_id'  (언더스코어)  ← key 이름 다름!

def split_value(log):
    d = ast.literal_eval(log['value'])#literal_eval : 문자열로 된 파이썬 자료형을 실제 자료형으로 변환
    if log['event'] == 'transaction':
        return pd.Series({
            'amount': d.get('amount'),
            'offer_id': None
        })
    else:
        # 두 가지 key 이름 모두 처리
        offer_id = d.get('offer id') or d.get('offer_id')
        return pd.Series({
            'amount': None,
            'offer_id': offer_id
        })

parsed = transcript.apply(split_value, axis=1)
transcript = pd.concat([transcript, parsed], axis=1)
transcript.drop(columns=['value'], inplace=True)

#time-> day
transcript['day'] = transcript['time'] // 24

transcript.head()

,person,event,time,amount,offer_id,day
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0


In [17]:
#-----------------------
# viewed 없이 completed만 있는 경우
#-----------------------

viewed_set = set(
    transcript[transcript['event'] == 'offer viewed']
    [['person', 'offer_id']]
    .apply(tuple, axis=1)
)

In [18]:
viewed_set

{('4aa52739e1e44cc48133aab0e9e594f8', '0b1e1539f2cc45b7b9fa7c272da2e1d7'),
 ('f875599b198d4e42b4c14c6e39e02b83', '4d5c57ea9a6940dd891ad53e9dbe8da0'),
 ('ce1d59f272f744c5aecefef2aba4d44f', '4d5c57ea9a6940dd891ad53e9dbe8da0'),
 ('f760c17c8e72437ab9498ec9a866e69a', '2298d6c36e964ae4a3e7e9706d1fb8c2'),
 ('32042db985ad426a8e3c792b1585291a', '5a8bc65990b245e5a138643cd4eb9837'),
 ('cb0147b0870d4aec9c0735188ed755fe', 'ae264e3637204a6fb9bb56bc8210ddfd'),
 ('c2e917c1c3fd4c839efe32d87667fc58', 'f19421c1d4aa40978ebb69ca19b0e20d'),
 ('4f74f4ade74b43bdb17c98a941c6b502', 'f19421c1d4aa40978ebb69ca19b0e20d'),
 ('85eeecda70f54cc2aace0bbdb1771293', '4d5c57ea9a6940dd891ad53e9dbe8da0'),
 ('97c8db12d9de44febcf5b937fe549be8', 'f19421c1d4aa40978ebb69ca19b0e20d'),
 ('6f8046e4e6c14ea699cfb964d67f5e4f', '5a8bc65990b245e5a138643cd4eb9837'),
 ('a295311c110841fab8833322d28f76c1', '0b1e1539f2cc45b7b9fa7c272da2e1d7'),
 ('e3dbe6d922674cf6b0d2054f8a6675f4', '2906b810c7d4411798c6938adc9daaa5'),
 ('3bc615dbfcdd40039113a5

In [19]:
# completed 이벤트에 대해, 해당 person과 offer_id가 viewed_set에 있는지 확인
def viewed_flag(log):
    if log['event'] != 'offer completed':
        return None
    return 1 if (log['person'], log['offer_id']) in viewed_set else 0

transcript['viewed_before_complete'] = transcript.apply(viewed_flag, axis=1)

In [20]:
unaware = transcript[transcript['viewed_before_complete'] == 0].shape[0]
total_completed = transcript[transcript['event'] == 'offer completed'].shape[0]
print(f" 이벤트 미인지 완료: {unaware}건 / 전체 완료: {total_completed}건 ({unaware/total_completed*100:.1f}%)")

 이벤트 미인지 완료: 4855건 / 전체 완료: 33579건 (14.5%)


In [21]:
# ----------------------
# 5. 데이터 병합
# ----------------------

#transcript + profile 병합 (person → id)
full_data = transcript.merge(
    profile.rename(columns={'id': 'person'}),
    on='person', 
    how='left')

#full_data + portfolio 병합 (offer_id → id)
full_data = full_data.merge(
    portfolio.rename(columns={'id': 'offer_id'}),
    on='offer_id',
    how='left')

print("✅ 데이터 병합 완료")
print(f"병합된 데이터 크기: {full_data.shape}")
full_data.head()

✅ 데이터 병합 완료
병합된 데이터 크기: (306534, 24)


,person,event,time,amount,offer_id,day,viewed_before_complete,gender,age,became_member_on,...,member_days_group,reward,difficulty,duration,offer_type,ch_web,ch_email,ch_mobile,ch_social,channel_count
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,NaN,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,NaN,F,75.0,2017-05-09,...,중간 (1~2년),5.0,5.0,7.0,bogo,1.0,1.0,1.0,0.0,3.0
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,NaN,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,NaN,NaN,NaN,2017-08-04,...,신규 (1년 미만),5.0,20.0,10.0,discount,1.0,1.0,0.0,0.0,2.0
2,e2127556f4f64592b11af22de27a7932,offer received,0,NaN,2906b810c7d4411798c6938adc9daaa5,0,NaN,M,68.0,2018-04-26,...,신규 (1년 미만),2.0,10.0,7.0,discount,1.0,1.0,1.0,0.0,3.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,NaN,fafdcd668e3743c1bb461111dcafc2a4,0,NaN,NaN,NaN,2017-09-25,...,신규 (1년 미만),2.0,10.0,10.0,discount,1.0,1.0,1.0,1.0,4.0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,NaN,4d5c57ea9a6940dd891ad53e9dbe8da0,0,NaN,NaN,NaN,2017-10-02,...,신규 (1년 미만),10.0,10.0,5.0,bogo,1.0,1.0,1.0,1.0,4.0


In [22]:
full_data.to_csv('../Soyun_EDA/full_data.csv', index=False)

In [23]:
full_data.dtypes

person                               str
event                                str
time                               int64
amount                           float64
offer_id                             str
day                                int64
viewed_before_complete           float64
gender                               str
age                              float64
became_member_on          datetime64[us]
income                           float64
member_days                        int64
age_group                            str
income_group                         str
member_days_group                    str
reward                           float64
difficulty                       float64
duration                         float64
offer_type                           str
ch_web                           float64
ch_email                         float64
ch_mobile                        float64
ch_social                        float64
channel_count                    float64
dtype: object